In [1]:
import os, requests
from dotenv import load_dotenv
import pandas as pd
import time
import csv
import os
import subprocess
import json
import numpy as np
from shutil import rmtree
# Load .env file
load_dotenv()

# Get token from environment
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("❌ Please set GITHUB_TOKEN in environment variables.")

# GitHub API headers
HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.mockingbird-preview+json, application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
version = "all"  # Change this as needed

In [2]:
import sys
print(sys.executable)

c:\Users\Xrayon14\Desktop\New folder\rep5\venv\Scripts\python.exe


In [3]:
import os

CACHE_DIR = "data_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

import pandas as pd

"""datasets = {
    "pr_df": "hf://datasets/hao-li/AIDev/pull_request.parquet",
    "pr_task_type_df": "hf://datasets/hao-li/AIDev/pr_task_type.parquet",
    "repo_df": "hf://datasets/hao-li/AIDev/repository.parquet",
    "pr_commits_df": "hf://datasets/hao-li/AIDev/pr_commits.parquet",
    "pr_commit_details_df": "hf://datasets/hao-li/AIDev/pr_commit_details.parquet",
    "human_pr_df": "hf://datasets/hao-li/AIDev/human_pull_request.parquet",
    "human_pr_task_type_df": "hf://datasets/hao-li/AIDev/human_pr_task_type.parquet",
    "pr_comments_df": "hf://datasets/hao-li/AIDev/pr_comments.parquet",
}

for name, path in datasets.items():
    print(f"Downloading {name}...")
    df = pd.read_parquet(path)
    
    save_path = os.path.join(CACHE_DIR, f"{name}.parquet")
    df.to_parquet(save_path, index=False)
    
print("Done saving all datasets locally.")
"""
import pandas as pd
import os

CACHE_DIR = "data_cache"

pr_df = pd.read_parquet(f"{CACHE_DIR}/pr_df.parquet")
pr_task_type_df = pd.read_parquet(f"{CACHE_DIR}/pr_task_type_df.parquet")
repo_df = pd.read_parquet(f"{CACHE_DIR}/repo_df.parquet")

pr_commits_df = pd.read_parquet(f"{CACHE_DIR}/pr_commits_df.parquet")
pr_commit_details_df = pd.read_parquet(f"{CACHE_DIR}/pr_commit_details_df.parquet")

human_pr_df = pd.read_parquet(f"{CACHE_DIR}/human_pr_df.parquet")
human_pr_task_type_df = pd.read_parquet(f"{CACHE_DIR}/human_pr_task_type_df.parquet")

pr_comments_df = pd.read_parquet(f"{CACHE_DIR}/pr_comments_df.parquet")

## Identifying the PRs with Fix types

In [4]:
print(len(pr_task_type_df))
fix_prs = pr_task_type_df[pr_task_type_df['type'] == 'fix']
print(len(fix_prs))
print("columns in fix_prs:", fix_prs.columns.tolist())
# print("number of repos:", len(set(fix_prs['repo_id'])))

33596
8106
columns in fix_prs: ['agent', 'id', 'title', 'reason', 'type', 'confidence']


## Separating the Python Repositories and their associated PRs

In [5]:
repo_df = repo_df[repo_df['language'] == 'Python']
print(len(repo_df))
python_repo_ids = set(repo_df['id'])
print(len(pr_df))

python_prs = pr_df[pr_df['repo_id'].isin(python_repo_ids)]
print(len(python_prs))

python_fix_prs = python_prs[python_prs['id'].isin(fix_prs['id'])]
print(len(python_fix_prs))
python_fix_prs = python_fix_prs[python_fix_prs['merged_at'].notnull()]
print(len(python_fix_prs))
# python_fix_prs.to_csv(f"1.python_fix_prs.csv", index=False)
print("number of repos:", len(set(python_fix_prs['repo_id'])))


530
33596
7191
1802
1210
number of repos: 206


## Analyzing each PR

In [ ]:
form pathlib import Path
REPO_DIR = Path.cwd().parent
SONAR_URL = "http://localhost:9000" 
SONAR_TOKEN = "put your tokken here" # Replace with your SonarQube token 
unanalyzed_prs = [] 
analyzed_results = [] # Store all PR analysis results 
import json 
import shutil 
# ------------------------------- # Helpers # ------------------------------- 
# Run a shell command in a specific directory
def run(cmd, cwd=REPO_DIR):
    result = subprocess.run(
        cmd,
        cwd=cwd,
        shell=True,
        text=True,
        capture_output=True
    )

    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise Exception(f"Command failed:\n{result.stderr}")


# Clone a git repository
def clone_repo(repo_url, repo_path):
    run(f"git clone {repo_url} \"{repo_path}\"", cwd=REPO_DIR)


# Extract parent and merged SHAs from PR commits
def extracting_PR_commits(pr_api_url):
    try:
        print("🔵 STEP 1: GitHub API")
        response = requests.get(pr_api_url, headers=HEADERS)
        response.raise_for_status()

        pr_data = response.json()
        merged_sha = pr_data['merge_commit_sha']
        base_sha = pr_data['base']['sha']

        return merged_sha, base_sha

    except requests.RequestException as e:
        with open(log_file, mode='a', encoding='utf-8') as log:
            log.write(f"Error occurred while fetching PR data: {e}\n")

        print(f"Error occurred while fetching PR data: {e}")
        return None, None
        

def checkout_main(repo_path):
    result = subprocess.run(
        "git symbolic-ref refs/remotes/origin/HEAD",
        cwd=repo_path,
        shell=True,
        capture_output=True,
        text=True
    )

    if "main" in result.stdout:
        run("git checkout main", cwd=repo_path)
    else:
        run("git checkout master", cwd=repo_path)
# -------------------------------
# SonarQube API helpers
# -------------------------------

def sonar_post(endpoint, params=None):
    r = requests.post(
        f"{SONAR_URL}{endpoint}",
        params=params,
        auth=(SONAR_TOKEN, "")
    )

    if r.status_code >= 400:
        print(r.text)
        r.raise_for_status()

    return r.json() if r.text else None


def sonar_get(endpoint, params=None):
    r = requests.get(
        f"{SONAR_URL}{endpoint}",
        params=params,
        auth=(SONAR_TOKEN, "")
    )

    if r.status_code >= 400:
        print(r.text)
        r.raise_for_status()

    return r.json()


# -------------------------------
# SonarQube project setup
# -------------------------------

def create_project(PROJECT_KEY, PROJECT_NAME):
    print("Creating SonarQube project...")
    print("🟡 STEP 2: Sonar API")

    sonar_post(
        "/api/projects/create",
        {
            "project": PROJECT_KEY,
            "name": PROJECT_NAME
        }
    )

    print("Configuring New Code period...")
    print("🟡 STEP 2: Sonar API")

    sonar_post(
        "/api/new_code_periods/set",
        {
            "project": PROJECT_KEY,
            "type": "PREVIOUS_VERSION"
        }
    )


# -------------------------------
# Sonar analysis
# -------------------------------

def analyze_commit(project_key, repo_path):
    print(f"Analyzing project {project_key} in {repo_path}")
    print("🟢 STEP 3: Sonar Scanner")

    run(
        f"sonar-scanner "
        f"-Dsonar.projectKey={project_key} "
        f"-Dsonar.sources=. "
        f"-Dsonar.inclusions=**/*.py "
        f"-Dsonar.host.url={SONAR_URL} "
        f"-Dsonar.login={SONAR_TOKEN}",
        cwd=repo_path
    )


# -------------------------------
# Cleanup
# -------------------------------

def delete_project(PROJECT_KEY):
    print("Deleting project...")
    print("🟡 STEP 2: Sonar API")

    sonar_post(
        "/api/projects/delete",
        {
            "project": PROJECT_KEY
        }
    )


# -------------------------------
# Logging
# -------------------------------

log_file = "process.log"

def log_message(message):
    with open(log_file, mode='a', encoding='utf-8') as log:
        log.write(message + "\n")

    print(message)
# -------------------------------
# Main processing
# -------------------------------
df = pd.read_csv("../dataset/python_fix_prs.csv")
print("Total Sample PRs: ", len(df))

for index, row in df.iterrows():
    pr_number = row['number']  
    pr_html = row['html_url']
    owner, repo = pr_html.split("/pull/")[0].replace(" ", "").split("/")[-2:]

    if repo == "azure-sdk-for-python":
        continue

    log_message(f"\n{'='*80}")
    log_message(f"Processing PR #{pr_number} from repository {owner}/{repo} (Index: {index})")

    repo_url = pr_html.split("/pull/")[0] + ".git"
    repo_path = os.path.join(REPO_DIR, repo)

    print(f"Cloning repository {owner}/{repo} from {repo_url} into {repo_path}...")

    # Clone if needed
    if not os.path.exists(repo_path):
        clone_repo(repo_url, repo_path)
    else:
        log_message(f"Repository {repo} already cloned.")

    pr_api_url = row['repo_url'] + f"/pulls/{pr_number}"
    merged_sha, base_sha = extracting_PR_commits(pr_api_url)

    log_message("\n-------------------------------")
    log_message(f"Base SHA: {base_sha}, Merged SHA: {merged_sha}")

    try:
        PROJECT_KEY = f"PR_{repo}_{pr_number}"
        PROJECT_NAME = f"PR_{repo}_{pr_number}"

        try:
            sonar_get(f"/api/projects/search", {"projects": PROJECT_KEY})
            log_message(f"Project {PROJECT_KEY} already exists. Deleting it...")
            delete_project(PROJECT_KEY)
            create_project(PROJECT_KEY, PROJECT_NAME)

        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 404:
                create_project(PROJECT_KEY, PROJECT_NAME)
            else:
                raise

        log_message("\n---------------------------------")

        # =========================
        # FIX: CLEAN GIT STATE
        # =========================
        checkout_main(repo_path)
        # always start clean state
        run("git fetch --all", cwd=repo_path)
        run("git reset --hard", cwd=repo_path)
        run("git clean -fdx", cwd=repo_path)
        # BASE commit branch (overwrite if exists)

        run("git status", cwd=repo_path)

        # BASE commit branch
        run(f"git fetch origin {base_sha}", cwd=repo_path)
        run(f"git checkout -B base_{pr_number} {base_sha}", cwd=repo_path)

        log_message(f"Checked out to branch base_{pr_number} at {base_sha}")
        analyze_commit(PROJECT_KEY, repo_path)

        # MERGED commit branch (overwrite if exists)
        checkout_main(repo_path)
        # always start clean state
        run("git fetch --all", cwd=repo_path)
        run("git reset --hard", cwd=repo_path)
        run("git clean -fdx", cwd=repo_path)
        run(f"git fetch origin {merged_sha}", cwd=repo_path)
        run(f"git checkout -B merged_{pr_number} {merged_sha}", cwd=repo_path)

        log_message(f"Checked out to branch merged_{pr_number} at {merged_sha}")
        analyze_commit(PROJECT_KEY, repo_path)
        time.sleep(2)

    except Exception as e:
        print(f"An error occurred while processing PR #{pr_number}: {str(e)}")
        print("Skipping to the next PR.")
        unanalyzed_prs.append(row)
        continue
if unanalyzed_prs:
    unanalyzed_prs_df = pd.DataFrame(unanalyzed_prs)
    unanalyzed_prs_df.to_csv('Unanalyzed_PRs.csv', index=False)
    print(f"✓ Saved {len(unanalyzed_prs)} unanalyzed PRs to 'Unanalyzed_PRs.csv'")
else:
    print("✓ All PRs were analyzed successfully!")
print(f"{'='*80}\n")

## Extract the analysis

In [ ]:
def fetch_new_issues(project_key):
    print("Fetching new issues (diff only)...")
    issues = []
    page = 1

    while True:
        data = sonar_get(
            "/api/issues/search",
            {
                "componentKeys": project_key,
                "sinceLeakPeriod": "true",
                "p": page,
                "ps": 500
            }
        )

        issues.extend(data.get("issues", []))

        if page * 500 >= data["paging"]["total"]:
            break
        page += 1

    print(f"Found {len(issues)} new issues")
    print(f"Issues details: {json.dumps(issues, indent=2)}")
    return issues

df = pd.read_csv("../dataset/python_fix_prs.csv")
print("Total Sample PRs: ", len(df))

for index, row in df.iterrows():
    pr_number = row['number']  # This is the actual PR number from GitHub
    pr_html = row['pr_html_url']
    owner, repo = pr_html.split("/pull/")[0].replace(" ", "").split("/")[-2:]

    PROJECT_KEY = f"PR_{repo}_{pr_number}"
    try:
        issues = fetch_new_issues(PROJECT_KEY)
    

    # Fetch project metrics
        
        print(f"Fetching analysis results for PR #{pr_number}...")
        print(f"{'-'*80}")
        print("Issues found:")
        for issue in issues:
            print(f"- {issue['key']}: {issue['message']} (Severity: {issue['severity']})")
        print(f"{'-'*80}")
            
        # Save metrics and PR info to JSON results
        analyzed_results.append({
                'pr_info': {
                    'pr_number': pr_number,
                    'pr_html_url': pr_html,
                    'owner': owner,
                    'repo': repo,
                    'Agent': row['agent']
                    #'base_sha': base_sha,
                    #'merged_sha': merged_sha
                },
                'issues_count': len(issues),
                'issues': issues
        })
            
        print(f"✓ Completed analysis for PR #{pr_number}\n")
        
    except Exception as e:
        print(f"Error occurred while fetching issues for PR #{pr_number}: {str(e)}")
        continue

# Save analyzed results to JSON
print(f"\n{'='*80}")
print(f"SAVING RESULTS TO JSON")
print(f"{'='*80}")

# Save as JSON file
with open(f'All_PR_Sonar_Results.json', 'w') as f:
    json.dump(analyzed_results, f, indent=2)
print(f"✓ Saved {len(analyzed_results)} PR metrics to 'All_PR_Sonar_Results.json'")

In [49]:
# Read the JSON file
with open(f'All_PR_Sonar_Results.json', 'r') as file:
    pr_sonar_results = json.load(file)

# Display the data (optional)
print(f"Loaded {len(pr_sonar_results)} PR results from 'All_PR_Sonar_Results.json'")
# Total number of instances in the dataset
total_instances = len(pr_sonar_results)
print(f"Total pull requests in the dataset: {total_instances}")
print("Number of PRs with issues > 0: ", sum(1 for pr in pr_sonar_results if pr.get('issues_count', 0) > 0))
total_issues = sum(pr.get('issues_count', 0) for pr in pr_sonar_results)
print(f"Total issues across all pull requests: {total_issues}")

# Convert pr_sonar_results to a list before saving as JSON
with open(f"All_PR_Sonar_Results.json", 'w') as json_file:
    json.dump(list(pr_sonar_results), json_file, indent=2)
# Count the number of issues by type
bug_issues = sum(
    1 for pr in pr_sonar_results for issue in pr.get('issues', []) if issue.get('type') == 'BUG'
)
code_smell_issues = sum(
    1 for pr in pr_sonar_results for issue in pr.get('issues', []) if issue.get('type') == 'CODE_SMELL'
)

print(f"Total BUG issues: {bug_issues}")
print(f"Total CODE_SMELL issues: {code_smell_issues}")


Loaded 1206 PR results from 'All_PR_Sonar_Results.json'
Total pull requests in the dataset: 1206
Number of PRs with issues > 0:  181
Total issues across all pull requests: 1107
Total BUG issues: 48
Total CODE_SMELL issues: 1059


# Create CSV from JSON

In [73]:
# Read the JSON file
with open(f'All_PR_Sonar_Results.json', 'r') as file:
    pr_sonar_results = json.load(file)

# Load the JSON data into a pandas DataFrame
df = pd.json_normalize(
        pr_sonar_results,
        # record_path='issues',
        meta=[
            ['pr_info', 'owner'],
            ['pr_info', 'repo'],
            ['pr_info', 'pr_number'],
            ['pr_info', 'Agent'],
            ['pr_info', 'pr_html_url']
        ],
        errors='ignore'
    )

# Rename columns for clarity
df.rename(columns={
        'pr_info.owner': 'Owner',
        'pr_info.repo': 'Repo',
        'pr_info.pr_number': 'PR Number',
        'pr_info.agent': 'Agent',
        'pr_info.pr_html_url': 'pr_html_url',
        'type': 'Issue Type',
        'severity': 'Severity',
        'message': 'Message',
        'component': 'File Name',
        'textRange.startLine': 'StartLine',
        'textRange.endLine': 'EndLine',
        'rule': 'Rule'
    }, inplace=True)

    # Save the DataFrame to a CSV file
df.to_csv("All_PR_Sonar_Results.csv", index=False)

# Define the CSV file name
output_csv = f"All_PR_Issues_Details.csv"
# Extract the required information and write to CSV
with open(output_csv, mode='w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['Owner/Repo', 'PR Number', 'Agent', 'html_url', 'Issue Type', 'Severity','Message', 'File Name', 'StartLine', 'EndLine', 'Rule']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for pr in pr_sonar_results:
        pr_number = pr['pr_info']['pr_number']
        issues = pr.get('issues', [])
        for issue in issues:
            writer.writerow({
                'Owner/Repo': f"{pr['pr_info']['owner']}/{pr['pr_info']['repo']}",
                'PR Number': pr_number,
                'Agent': pr['pr_info']['agent'],
                'html_url': pr['pr_info']['pr_html_url'],
                'Issue Type': issue.get('type', 'N/A'),
                'Severity': issue.get('severity', 'N/A'),
                'Message': issue.get('message', 'N/A'),
                'File Name': issue.get('component', 'N/A'),
                'StartLine': issue.get('textRange', {}).get('startLine', 'N/A'),
                'EndLine': issue.get('textRange', {}).get('endLine', 'N/A'),
                'Rule': issue.get('rule', 'N/A')
            })

print(f"CSV file '{output_csv}' has been created successfully.")

CSV file 'All_PR_Issues_Details.csv' has been created successfully.


# Fetch PR Additions and Deletions from GitHub

In [ ]:
import pandas as pd
import requests
import time
from urllib.parse import urlparse

# Load the PR data
df_prs = pd.read_csv(f'{REPO_DIR}/dataset/python_fix_prs_with_stats.csv')
df_prs = df_prs[df_prs['Issue Type']=='SECURITY_HOTSPOT']
print(f"Total PRs to process: {len(df_prs)}")
print(f"Columns: {df_prs.columns.tolist()}")
print()

# Function to extract owner and repo from html_url
def parse_github_url(html_url):
    """
    Extract owner and repo from GitHub PR URL
    Example: https://github.com/owner/repo/pull/123 -> (owner, repo, 123)
    """
    try:
        parts = html_url.rstrip('/').split('/')
        owner = parts[-4]
        repo = parts[-3]
        pr_number = parts[-1]
        return owner, repo, pr_number
    except:
        return None, None, None

# Function to get PR additions and deletions from GitHub API
def get_pr_stats(owner, repo, pr_number, headers):
    """
    Fetch PR statistics (additions, deletions, changed_files) from GitHub API
    """
    url = f"https://api.github.com/repos/{owner}/{repo}/pulls/{pr_number}"
    
    try:
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            return {
                'additions': data.get('additions', 0),
                'deletions': data.get('deletions', 0),
                'changed_files': data.get('changed_files', 0),
                'changed_loc': data.get('additions', 0) + data.get('deletions', 0)
            }
        elif response.status_code == 404:
            print(f"  ⚠ PR not found: {owner}/{repo}#{pr_number}")
            return None
        elif response.status_code == 403:
            print(f"  ⚠ Rate limit exceeded. Waiting...")
            time.sleep(60)  # Wait for 1 minute
            return get_pr_stats(owner, repo, pr_number, headers)  # Retry
        else:
            print(f"  ⚠ Error {response.status_code} for {owner}/{repo}#{pr_number}")
            return None
    except Exception as e:
        print(f"  ❌ Exception for {owner}/{repo}#{pr_number}: {e}")
        return None

# Prepare results storage
results = []

# Process each PR
print("Fetching PR statistics from GitHub API...")
print("="*100)

for idx, row in df_prs.iterrows():
    html_url = row['html_url']
    print(f"Processing PR {idx + 1}/{len(df_prs)}: {html_url}")
    # Parse URL
    owner, repo, pr_number = parse_github_url(html_url)
    print(f"  Parsed: owner={owner}, repo={repo}, pr_number={pr_number}")
    if owner is None:
        print(f"⚠ Could not parse URL: {html_url}")
        results.append({
            'id': row['id'],
            'number': row['number'],
            'html_url': html_url,
            'agent': row['agent'],
            'additions': None,
            'deletions': None,
            'changed_files': None,
            'changed_loc': None
        })
        continue
    
    # Get PR stats
    stats = get_pr_stats(owner, repo, pr_number, HEADERS)
    
    if stats:
        results.append({
            'id': row['id'],
            'number': row['number'],
            'html_url': html_url,
            'agent': row['agent'],
            'additions': stats['additions'],
            'deletions': stats['deletions'],
            'changed_files': stats['changed_files'],
            'changed_loc': stats['changed_loc']
        })
        
        # Progress indicator
        if (idx + 1) % 50 == 0:
            print(f"Progress: {idx + 1}/{len(df_prs)} PRs processed")
    else:
        results.append({
            'id': row['id'],
            'number': row['number'],
            'html_url': html_url,
            'agent': row['agent'],
            'additions': None,
            'deletions': None,
            'changed_files': None,
            'changed_loc': None
        })
    
    # Rate limiting: Sleep to avoid hitting API limits (5000 requests/hour)
    time.sleep(0.75)  # ~0.75 seconds = ~4800 requests/hour

print()
print("="*100)
print("Fetching complete!")
print()

# Create DataFrame with results
df_pr_stats = pd.DataFrame(results)

# Display summary
print("="*100)
print("SUMMARY OF PR STATISTICS")
print("="*100)
print(f"Total PRs processed: {len(df_pr_stats)}")
print(f"PRs with data: {df_pr_stats['additions'].notna().sum()}")
print(f"PRs with missing data: {df_pr_stats['additions'].isna().sum()}")
print()

# Statistics by agent
print("Statistics by Agent:")
print(df_pr_stats.groupby('agent').agg({
    'changed_loc': ['count', 'mean', 'median', 'min', 'max'],
    'additions': ['mean', 'median'],
    'deletions': ['mean', 'median']
}).round(2))
print()

# Save to CSV
output_file = 'test.csv'
df_pr_stats.to_csv(output_file, index=False)
print(f"✓ Results saved to: {output_file}")
print()

# Display first few rows
print("First 10 rows:")
print(df_pr_stats.head(10))
print()
print("="*100)

# Create Combined Dataset: Merge PR Info with LOC Stats

In [74]:
import pandas as pd

# Load both datasets
print("Loading datasets...")
df_issues = pd.read_csv('All_PR_Issues_Details.csv')
df_stats = pd.read_csv(f'{REPO_DIR}/dataset/python_fix_prs_with_stats.csv')
df_stats.rename(columns={'agent': 'Agent'}, inplace=True)
print(f"✓ Loaded 6.All_PR_Issues_Details.csv: {len(df_issues)} rows, {len(df_issues.columns)} columns")
print(f"✓ Loaded 7.python_fix_prs_with_stats.csv: {len(df_stats)} rows, {len(df_stats.columns)} columns")
print()

# Check columns
print("Columns in 6.All_PR_Issues_Details.csv:")
print(df_issues.columns.tolist())
print()

print("Columns in 7.python_fix_prs_with_stats.csv:")
print(df_stats.columns.tolist())
print()

# Merge on html_url
print("Merging datasets on 'html_url'...")
df_combined = df_issues.merge(
    df_stats[['html_url', 'additions', 'deletions', 'changed_files', 'changed_loc']],
    on='html_url',
    how='left'
)

print(f"✓ Merged dataset: {len(df_combined)} rows, {len(df_combined.columns)} columns")
print()

# Verify the merge
print("="*100)
print("MERGE VERIFICATION")
print("="*100)
print()

print(f"Total rows: {len(df_combined)}")
print(f"Rows with LOC data: {df_combined['changed_loc'].notna().sum()}")
print(f"Rows without LOC data: {df_combined['changed_loc'].isna().sum()}")
print()

# Check by agent
print("LOC data availability by agent:")

merge_check = df_combined.groupby('Agent').agg({
    'html_url': 'count',
    'changed_loc': lambda x: x.notna().sum()
}).rename(columns={'html_url': 'Total_Rows', 'changed_loc': 'Rows_with_LOC'})
merge_check['Missing_LOC'] = merge_check['Total_Rows'] - merge_check['Rows_with_LOC']
print(merge_check)
print()

# Show sample of merged data
print("Sample of merged data (first 5 rows):")
display_cols = ['Owner/Repo', 'PR Number', 'Agent', 'html_url', 'Issue Type', 'Severity', 'additions', 'deletions', 'changed_files', 'changed_loc']
print(df_combined[display_cols].head())
print()

# Save the combined dataset
output_file = 'All_PR_Issues_Details_with_LOC.csv'
df_combined.to_csv(output_file, index=False)
print(f"✓ Combined dataset saved to: {output_file}")
print()

# Check unique PRs with missing LOC data
unique_prs_missing = df_combined[df_combined['changed_loc'].isna()]['html_url'].nunique()
if unique_prs_missing > 0:
    print(f"⚠ WARNING: {unique_prs_missing} unique PR(s) have missing LOC data:")
    missing_prs = df_combined[df_combined['changed_loc'].isna()][['Agent', 'html_url']].drop_duplicates()
    print(missing_prs.head(10))
    if len(missing_prs) > 10:
        print(f"... and {len(missing_prs) - 10} more")
else:
    print("✓ All PRs have LOC data!")

print()
print("="*100)
print(f"COMBINED DATASET CREATED: {output_file}")
print(f"This dataset contains issue details with PR size metrics (LOC)")
print("="*100)

Loading datasets...
✓ Loaded 6.All_PR_Issues_Details.csv: 1107 rows, 11 columns
✓ Loaded 7.python_fix_prs_with_stats.csv: 1211 rows, 8 columns

Columns in 6.All_PR_Issues_Details.csv:
['Owner/Repo', 'PR Number', 'Agent', 'html_url', 'Issue Type', 'Severity', 'Message', 'File Name', 'StartLine', 'EndLine', 'Rule']

Columns in 7.python_fix_prs_with_stats.csv:
['id', 'number', 'html_url', 'Agent', 'additions', 'deletions', 'changed_files', 'changed_loc']

Merging datasets on 'html_url'...
✓ Merged dataset: 1107 rows, 15 columns

MERGE VERIFICATION

Total rows: 1107
Rows with LOC data: 1106
Rows without LOC data: 1

LOC data availability by agent:
              Total_Rows  Rows_with_LOC  Missing_LOC
Agent                                               
Claude_Code           62             62            0
Copilot              224            223            1
Cursor               329            329            0
Devin                 82             82            0
OpenAI_Codex         410      

# Extracting the Security Hotspots from the PRs

In [ ]:
REPO_DIR = "/home/uji657/Downloads/PhD_Projects/OTHERS/MSR-Mining-2026"
SONAR_URL = "http://localhost:9000"
SONAR_TOKEN = "squ_f684e08d624d4870b6eaa904d07198c78e9b3635"

def sonar_get(endpoint, params=None):
    r = requests.get(
        f"{SONAR_URL}{endpoint}",
        params=params,
        auth=(SONAR_TOKEN, "")
    )
    if r.status_code >= 400:
        print(r.text)
        r.raise_for_status()
    return r.json()

def fetch_security_hotspots(project_key):
    """
    Fetch security hotspots for a given project from SonarQube API
    """
    print(f"Fetching security hotspots for project {project_key}...")
    hotspots = []
    page = 1
    
    while True:
        try:
            data = sonar_get(
                "/api/hotspots/search",
                {
                    "projectKey": project_key,
                    "inNewCodePeriod": "true",
                    "p": page,
                    "ps": 500
                }
            )
            
            hotspots.extend(data.get("hotspots", []))
            
            if page * 500 >= data.get("paging", {}).get("total", 0):
                break
            page += 1
            
        except Exception as e:
            print(f"Error fetching hotspots for {project_key}: {str(e)}")
            break
    
    print(f"Found {len(hotspots)} security hotspots")
    return hotspots

# Load the PR data
df = pd.read_csv("python_fix_prs.csv")
# df = df.iloc[:5]
print(f"Total PRs to process: {len(df)}")

# Store results
hotspot_results = []

for index, row in df.iterrows():
    pr_number = row['number']
    pr_html = row['html_url']
    owner, repo = pr_html.split("/pull/")[0].replace(" ", "").split("/")[-2:]
    
    PROJECT_KEY = f"PR_{repo}_{pr_number}"
    
    print(f"\n{'='*80}")
    print(f"Processing PR #{pr_number} from {owner}/{repo} (Index: {index})")
    print(f"{'='*80}")
    
    # Fetch security hotspots
    hotspots = fetch_security_hotspots(PROJECT_KEY)
    if hotspots:
        # Store results
        hotspot_results.append({
            'pr_info': {
                'pr_number': pr_number,
                'pr_html_url': pr_html,
                'owner': owner,
                'repo': repo,
                'agent': row['agent']
            },
            'hotspots_count': len(hotspots),
            'hotspots': hotspots
        })
    
    print(f"✓ Found {len(hotspots)} security hotspots for PR #{pr_number}")

# Save results to JSON
output_file = 'Security_Hotspots.json'
with open(output_file, 'w') as f:
    json.dump(hotspot_results, f, indent=2)

print(f"\n{'='*80}")
print(f"✓ Saved {len(hotspot_results)} PR security hotspot results to '{output_file}'")
print(f"{'='*80}")

# Summary statistics
total_hotspots = sum(pr['hotspots_count'] for pr in hotspot_results)
prs_with_hotspots = sum(1 for pr in hotspot_results if pr['hotspots_count'] > 0)

print(f"\nSummary:")
print(f"Total PRs analyzed: {len(hotspot_results)}")
print(f"PRs with security hotspots: {prs_with_hotspots}")
print(f"Total security hotspots: {total_hotspots}")

# Convert security hotspot json result to csv

In [ ]:
# Load the security hotspots JSON file
with open('test.json', 'r') as file:
    hotspot_data = json.load(file)

# Define the CSV file name
output_csv = "8.Security_Hotspots_Details.csv"

# Extract the required information and write to CSV
with open(output_csv, mode='w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['Owner/Repo', 'PR Number', 'Agent', 'html_url', 'Issue Type', 'Severity', 'Message', 'File Name', 'StartLine', 'EndLine', 'Rule']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for pr in hotspot_data:
        pr_number = pr['pr_info']['pr_number']
        pr_html_url = pr['pr_info']['pr_html_url']
        owner = pr['pr_info']['owner']
        repo = pr['pr_info']['repo']
        agent = pr['pr_info']['agent']
        hotspots = pr.get('hotspots', [])
        
        for hotspot in hotspots:
            writer.writerow({
                'Owner/Repo': f"{owner}/{repo}",
                'PR Number': pr_number,
                'Agent': agent,
                'html_url': pr_html_url,
                'Issue Type': 'SECURITY_HOTSPOT',
                'Severity': hotspot.get('vulnerabilityProbability', 'N/A'),
                'Message': hotspot.get('message', 'N/A'),
                'File Name': hotspot.get('component', 'N/A'),
                'StartLine': hotspot.get('textRange', {}).get('startLine', 'N/A'),
                'EndLine': hotspot.get('textRange', {}).get('endLine', 'N/A'),
                'Rule': hotspot.get('securityCategory', 'N/A')
            })

print(f"CSV file '{output_csv}' has been created successfully.")
print(f"Total security hotspots exported: {sum(pr['hotspots_count'] for pr in hotspot_data)}")

# Merge Security Hotspots with PR Sonar Results

In [ ]:
import pandas as pd
import json

# Load the security hotspots JSON file
with open('8.Security_Hotspots.json', 'r') as f:
    security_hotspots_data = json.load(f)

# Load the All PR Sonar Results CSV
df_sonar_results = pd.read_csv('5.All_PR_Sonar_Results2.csv')

print(f"Loaded {len(security_hotspots_data)} security hotspot entries")
print(f"Loaded {len(df_sonar_results)} rows from Sonar Results CSV")
print()

# Create a dictionary mapping html_url to security hotspots
hotspots_by_url = {}
for entry in security_hotspots_data:
    pr_url = entry['pr_info']['html_url']
    hotspots_by_url[pr_url] = entry['hotspots']

print(f"Created mapping for {len(hotspots_by_url)} unique PR URLs with security hotspots")
print()

# Function to merge security hotspots with existing issues
def merge_security_hotspots(row):
    pr_url = row['html_url']
    
    # Get existing issues (convert from string representation to list)
    try:
        existing_issues = eval(row['issues']) if pd.notna(row['issues']) and row['issues'] != '[]' else []
    except:
        existing_issues = []
    
    # Get security hotspots for this PR
    hotspots = hotspots_by_url.get(pr_url, [])
    
    # Convert security hotspots to issue format and add to existing issues
    for hotspot in hotspots:
        issue = {
            'key': hotspot['key'],
            'rule': hotspot.get('securityCategory', 'N/A'),
            'severity': hotspot.get('vulnerabilityProbability', 'N/A'),
            'component': hotspot['component'],
            'project': hotspot['project'],
            'line': hotspot.get('line'),
            'message': hotspot['message'],
            'author': hotspot.get('author'),
            'creationDate': hotspot.get('creationDate'),
            'updateDate': hotspot.get('updateDate'),
            'type': 'SECURITY_HOTSPOT',
            'status': hotspot.get('status'),
            'textRange': hotspot.get('textRange', {}),
            'flows': hotspot.get('flows', [])
        }
        existing_issues.append(issue)
    
    # Update the row
    new_issues_count = row['issues_count'] + len(hotspots)
    
    return pd.Series({
        'issues_count': new_issues_count,
        'issues': str(existing_issues)
    })

# Apply the merge function
print("Merging security hotspots with existing issues...")
df_sonar_results[['issues_count', 'issues']] = df_sonar_results.apply(merge_security_hotspots, axis=1)

print(f"✓ Merge complete!")
print()

# Display summary
print("="*80)
print("SUMMARY")
print("="*80)
print(f"Total rows: {len(df_sonar_results)}")
print(f"Total issues (including security hotspots): {df_sonar_results['issues_count'].sum()}")
print(f"Rows with issues > 0: {(df_sonar_results['issues_count'] > 0).sum()}")
print()

# Count issues by type (requires parsing)
def count_issue_types(df):
    type_counts = {'BUG': 0, 'CODE_SMELL': 0, 'SECURITY_HOTSPOT': 0, 'OTHER': 0}
    
    for _, row in df.iterrows():
        try:
            issues = eval(row['issues']) if pd.notna(row['issues']) and row['issues'] != '[]' else []
            for issue in issues:
                issue_type = issue.get('type', 'OTHER')
                if issue_type in type_counts:
                    type_counts[issue_type] += 1
                else:
                    type_counts['OTHER'] += 1
        except:
            continue
    
    return type_counts

type_counts = count_issue_types(df_sonar_results)
print("Issue counts by type:")
for issue_type, count in type_counts.items():
    print(f"  {issue_type}: {count}")
print()

# Save the updated CSV
output_file = '5.All_PR_Sonar_Results.csv'
# df_sonar_results.to_csv(output_file, index=False)
print(f"✓ Updated results saved to: {output_file}")
print("="*80)
print("number of PRs having issues>0 across agents: ", (df_sonar_results['issues_count'] > 0).sum())